In [0]:
-- 1. Create a Stateful Dictionary for Imputation
-- This Materialized View safely computes the max region per district over the entire Bronze table
CREATE MATERIALIZED VIEW dict_sales_regions
COMMENT "Dictionary to map sales_district to a known sales_region for imputation."
AS SELECT 
  TRIM(sales_district) AS sales_district, 
  MAX(TRIM(sales_region)) AS imputed_region
FROM mavenmarket.bronze.regions
WHERE sales_district IS NOT NULL AND sales_region IS NOT NULL
GROUP BY TRIM(sales_district);

-- 2. Read, Type, Trim, Impute (via Stream-Static Join), and generate Surrogate Key
-- Note: DISTINCT is removed; APPLY CHANGES INTO natively handles deduplication on the primary key.
CREATE TEMPORARY STREAMING TABLE stg_regions_typed AS
SELECT 
  -- Generate a unique Surrogate Key for this specific version of the region record
  md5(concat_ws('|', TRIM(r.region_id), CAST(r.ingested_timestamp AS STRING))) AS region_sk,
  
  CAST(TRIM(r.region_id) AS INT) AS region_id,
  TRIM(r.sales_district) AS sales_district,
  
  -- Impute missing sales_region using the Stream-Static Join against our dictionary
  COALESCE(
    TRIM(r.sales_region),
    m.imputed_region
  ) AS sales_region,
  
  r.ingested_timestamp,
  r.batch_date
FROM STREAM(mavenmarket.bronze.regions) r
LEFT JOIN LIVE.dict_sales_regions m 
  ON TRIM(r.sales_district) = m.sales_district;

-- 3. Run Generic KPI Checks (Completeness, Validity)
CREATE TEMPORARY STREAMING TABLE stg_regions_checked AS
SELECT 
  *,
  -- HARD CHECKS (Route to Quarantine)
  CASE WHEN region_id IS NULL THEN 'Completeness: region_id is null; ' ELSE '' END AS err_region_id,
  
  -- WARN CHECKS (Stay in Silver, added to dq_flags)
  CASE WHEN sales_district IS NULL THEN 'Completeness: sales_district is null; ' ELSE '' END AS warn_district,
  CASE WHEN sales_region IS NULL THEN 'Completeness: sales_region could not be imputed and is null; ' ELSE '' END AS warn_region
FROM STREAM(LIVE.stg_regions_typed);

-- 4. Consolidate Flags
CREATE TEMPORARY STREAMING TABLE stg_regions_flagged AS
SELECT 
  *,
  TRIM(CONCAT(err_region_id)) AS hard_failures,
  TRIM(CONCAT(warn_district, warn_region)) AS dq_flags
FROM STREAM(LIVE.stg_regions_checked);

-- 5. Quarantine Table Setup (Records failing hard checks)
CREATE STREAMING TABLE quarantine_regions
COMMENT "Quarantined regions failing hard checks (e.g. missing region_id)"
AS SELECT 
  * EXCEPT(err_region_id, warn_district, warn_region, dq_flags, hard_failures),
  hard_failures AS quarantine_reasons
FROM STREAM(LIVE.stg_regions_flagged)
WHERE hard_failures != '';

-- 6. Valid Regions Stream Setup (Records passing hard checks)
CREATE TEMPORARY STREAMING TABLE stg_regions_valid AS
SELECT 
  * EXCEPT(err_region_id, warn_district, warn_region, hard_failures)
FROM STREAM(LIVE.stg_regions_flagged)
WHERE hard_failures = '';

-- 7. Target Silver Table Declaration & SCD Type 2 Implementation
CREATE STREAMING TABLE silver_regions
COMMENT "Cleaned regions dimension with imputed sales regions and warn-level DQ flags. Managed as an SCD Type 2 Dimension.";

APPLY CHANGES INTO LIVE.silver_regions
FROM STREAM(LIVE.stg_regions_valid)
KEYS (region_id)
SEQUENCE BY ingested_timestamp
STORED AS SCD TYPE 2;